###Business Problem Statement:-

The organization is experiencing a noticeable increase in employee attrition, which is leading to:


•	Increased hiring and onboarding costs

•	Loss of experienced workforce

•	Reduced productivity and team stability

•	Impact on overall business performance

The objective is to build a predictive system that can identify employees at risk of attrition and provide actionable insights to the Human Resources (HR) team for timely intervention.


In [84]:
# Import Libraries
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import (
    confusion_matrix, accuracy_score,
    precision_score, recall_score, f1_score,
    classification_report
)

warnings.filterwarnings('ignore')


In [85]:
# folder to save all plots
os.makedirs("output_plots", exist_ok=True)

In [86]:
# Load Dataset

df = pd.read_csv("/content/sample_data/HR-Employee-Attrition.csv")
print("Dataset loaded. Shape:", df.shape)

Dataset loaded. Shape: (1470, 35)


In [87]:
# EDA - Exploratory Data Analysis
# We look at the data to understand its structure before building any model

# shappe returns the number of row and column counts
print("Shape:", df.shape)


Shape: (1470, 35)


In [88]:
#prints information about a DataFrame including the index dtype and columns, non-NA values and memory usage

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

In [89]:
# generates descriptive statistics
print(df.describe().T)

                           count          mean          std     min      25%  \
Age                       1470.0     36.923810     9.135373    18.0    30.00   
DailyRate                 1470.0    802.485714   403.509100   102.0   465.00   
DistanceFromHome          1470.0      9.192517     8.106864     1.0     2.00   
Education                 1470.0      2.912925     1.024165     1.0     2.00   
EmployeeCount             1470.0      1.000000     0.000000     1.0     1.00   
EmployeeNumber            1470.0   1024.865306   602.024335     1.0   491.25   
EnvironmentSatisfaction   1470.0      2.721769     1.093082     1.0     2.00   
HourlyRate                1470.0     65.891156    20.329428    30.0    48.00   
JobInvolvement            1470.0      2.729932     0.711561     1.0     2.00   
JobLevel                  1470.0      2.063946     1.106940     1.0     1.00   
JobSatisfaction           1470.0      2.728571     1.102846     1.0     2.00   
MonthlyIncome             1470.0   6502.

In [90]:
print(df.head())

   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences              1               1   
1                 8          1  Life Sciences              1               2   
2                 2          2          Other              1               4   
3                 3          4  Life Sciences              1               5   
4                 2          1        Medical              1               7   

   ...  RelationshipSatisfaction StandardHours  StockOptionLevel  \
0  ...

In [91]:
##Dropping useless columns
df.drop(columns=['EmployeeNumber', 'EmployeeCount', 'Over18', 'StandardHours'], inplace=True)
print("Remaining columns:", df.shape[1])

Remaining columns: 31


In [92]:
# Check for null/missing values

print(df.isnull().sum())

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
dtype: int64


In [93]:
# Check for duplicate rows

print("Duplicate rows:", df.duplicated().sum())
df.drop_duplicates(inplace=True)

Duplicate rows: 0


In [94]:
# Separate categorical and numerical columns
cat_cols = df.select_dtypes(include="object").columns.tolist()
num_cols = df.select_dtypes(exclude="object").columns.tolist()
print("\nCategorical columns:", cat_cols)
print("Numerical columns:", num_cols)


Categorical columns: ['Attrition', 'BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Numerical columns: ['Age', 'DailyRate', 'DistanceFromHome', 'Education', 'EnvironmentSatisfaction', 'HourlyRate', 'JobInvolvement', 'JobLevel', 'JobSatisfaction', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked', 'PercentSalaryHike', 'PerformanceRating', 'RelationshipSatisfaction', 'StockOptionLevel', 'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance', 'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion', 'YearsWithCurrManager']


In [95]:
# for categorical(text) columns: Use mode
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

# for discrete number columns: use median
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

In [96]:
# Check for error values like ?, @, etc.

error_vals = ['?', '@', '#', 'NA', 'N/A', 'none']
for col in cat_cols:
    found = df[col].isin(error_vals).sum()
    if found > 0:
        print(col, ":", found, "error values found, replacing...")
        df[col].replace(error_vals, np.nan, inplace=True)
        df[col].fillna(df[col].mode()[0], inplace=True)


In [97]:
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EnvironmentSatisfaction,Gender,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,2,Female,...,3,1,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,3,Male,...,4,4,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,4,Male,...,3,2,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,4,Female,...,3,3,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,Male,...,3,4,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,3,Male,...,3,3,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,4,Male,...,3,1,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,2,Male,...,4,2,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,4,Male,...,3,4,0,17,3,2,9,6,0,8


In [98]:
# Outlier detection using IQR method
print("\n--- Outlier Check (IQR Method) ---")
for col in num_cols:
    Q1    = df[col].quantile(0.25)
    Q3    = df[col].quantile(0.75)
    IQR   = Q3 - Q1
    LB    = Q1 - 1.5 * IQR   # lower boundary
    UB    = Q3 + 1.5 * IQR   # upper boundary
    count = df[(df[col] < LB) | (df[col] > UB)].shape[0]
    if count > 0:
        print("  " + col + ":", count, "outliers")



--- Outlier Check (IQR Method) ---
  MonthlyIncome: 114 outliers
  NumCompaniesWorked: 52 outliers
  PerformanceRating: 226 outliers
  StockOptionLevel: 85 outliers
  TotalWorkingYears: 63 outliers
  TrainingTimesLastYear: 238 outliers
  YearsAtCompany: 104 outliers
  YearsInCurrentRole: 21 outliers
  YearsSinceLastPromotion: 107 outliers
  YearsWithCurrManager: 14 outliers


In [99]:
# boxplot for the 4 columns mentioned in the Final Capstone Project document
# these are also the columns with the most outliers in the dataset
plt.figure(figsize=(16, 6))

plt.subplot(1, 4, 1)
plt.boxplot(df['Age'])
plt.title('Age')

plt.subplot(1, 4, 2)
plt.boxplot(df['MonthlyIncome'])
plt.title('MonthlyIncome')

plt.subplot(1, 4, 3)
plt.boxplot(df['TotalWorkingYears'])
plt.title('TotalWorkingYears')

plt.subplot(1, 4, 4)
plt.boxplot(df['YearsAtCompany'])
plt.title('YearsAtCompany')

plt.suptitle("Boxplots - Outlier Check")
plt.tight_layout()
plt.savefig("output_plots/01_outlier_boxplots.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 01_outlier_boxplots.png")


[Saved] 01_outlier_boxplots.png


In [100]:
# Target
### Target Variable: Attrition
print(df['Attrition'].value_counts())

Attrition
No     1233
Yes     237
Name: count, dtype: int64


In [101]:
plt.figure(figsize=(6, 4))
df['Attrition'].value_counts().plot(kind='bar', color=['steelblue', 'tomato'], edgecolor='black')
plt.title("Attrition Distribution")
plt.xlabel("Attrition")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("output_plots/02_target_distribution.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 02_target_distribution.png")


[Saved] 02_target_distribution.png


In [102]:
# Univariate analysis: histograms for numerical columns
plt.figure(figsize=(16, 12))

plt.subplot(3, 3, 1)
plt.hist(df['Age'], bins=20, color='steelblue', edgecolor='white')
plt.title('Age')

plt.subplot(3, 3, 2)
plt.hist(df['MonthlyIncome'], bins=20, color='steelblue', edgecolor='white')
plt.title('MonthlyIncome')

plt.subplot(3, 3, 3)
plt.hist(df['TotalWorkingYears'], bins=20, color='steelblue', edgecolor='white')
plt.title('TotalWorkingYears')

plt.subplot(3, 3, 4)
plt.hist(df['YearsAtCompany'], bins=20, color='steelblue', edgecolor='white')
plt.title('YearsAtCompany')

plt.subplot(3, 3, 5)
plt.hist(df['DistanceFromHome'], bins=20, color='steelblue', edgecolor='white')
plt.title('DistanceFromHome')

plt.subplot(3, 3, 6)
plt.hist(df['NumCompaniesWorked'], bins=20, color='steelblue', edgecolor='white')
plt.title('NumCompaniesWorked')

plt.subplot(3, 3, 7)
plt.hist(df['PercentSalaryHike'], bins=20, color='steelblue', edgecolor='white')
plt.title('PercentSalaryHike')

plt.subplot(3, 3, 8)
plt.hist(df['TrainingTimesLastYear'], bins=10, color='steelblue', edgecolor='white')
plt.title('TrainingTimesLastYear')

plt.subplot(3, 3, 9)
plt.hist(df['YearsSinceLastPromotion'], bins=15, color='steelblue', edgecolor='white')
plt.title('YearsSinceLastPromotion')

plt.suptitle("Univariate Analysis - Numerical Features", fontsize=14)
plt.tight_layout()
plt.savefig("output_plots/03_univariate_numerical.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 03_univariate_numerical.png")


[Saved] 03_univariate_numerical.png


In [103]:
# bar charts for categorical columns
plt.figure(figsize=(16, 9))

plt.subplot(2, 3, 1)
df['Department'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Department')
plt.xticks(rotation=30)

plt.subplot(2, 3, 2)
df['JobRole'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('JobRole')
plt.xticks(rotation=30)

plt.subplot(2, 3, 3)
df['MaritalStatus'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('MaritalStatus')
plt.xticks(rotation=0)

plt.subplot(2, 3, 4)
df['BusinessTravel'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('BusinessTravel')
plt.xticks(rotation=20)

plt.subplot(2, 3, 5)
df['Gender'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('Gender')
plt.xticks(rotation=0)

plt.subplot(2, 3, 6)
df['OverTime'].value_counts().plot(kind='bar', color='steelblue', edgecolor='black')
plt.title('OverTime')
plt.xticks(rotation=0)

plt.suptitle("Univariate Analysis - Categorical Features", fontsize=14)
plt.tight_layout()
plt.savefig("output_plots/04_univariate_categorical.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 04_univariate_categorical.png")


[Saved] 04_univariate_categorical.png


In [104]:
# Bivariate analysis: how each feature relates to Attrition
# This tells us WHY employees are leaving - useful for business recommendations
print("\n--- Bivariate Analysis (Feature vs Attrition) ---")

no_attr  = df[df['Attrition'] == 'No']
yes_attr = df[df['Attrition'] == 'Yes']

# attrition rate by categorical features
plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
ct = df.groupby('Department')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by Department')
plt.ylabel('%')
plt.xticks(rotation=20)

plt.subplot(2, 3, 2)
ct = df.groupby('MaritalStatus')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by MaritalStatus')
plt.ylabel('%')
plt.xticks(rotation=0)

plt.subplot(2, 3, 3)
ct = df.groupby('BusinessTravel')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by BusinessTravel')
plt.ylabel('%')
plt.xticks(rotation=20)

plt.subplot(2, 3, 4)
ct = df.groupby('Gender')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by Gender')
plt.ylabel('%')
plt.xticks(rotation=0)

plt.subplot(2, 3, 5)
ct = df.groupby('OverTime')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by OverTime')
plt.ylabel('%')
plt.xticks(rotation=0)

plt.subplot(2, 3, 6)
ct = df.groupby('JobRole')['Attrition'].value_counts(normalize=True).unstack().fillna(0) * 100
ct.plot(kind='bar', ax=plt.gca(), color=['steelblue', 'tomato'], edgecolor='black')
plt.title('Attrition by JobRole')
plt.ylabel('%')
plt.xticks(rotation=30)

plt.title("Bivariate Analysis - Attrition % by Categorical Features", fontsize=14)
plt.tight_layout()
plt.savefig("output_plots/05_bivariate_categorical_vs_attrition.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 05_bivariate_categorical_vs_attrition.png")



--- Bivariate Analysis (Feature vs Attrition) ---
[Saved] 05_bivariate_categorical_vs_attrition.png


In [105]:
# Histogram:-  numerical features vs Attrition
plt.figure(figsize=(16, 10))

plt.subplot(2, 3, 1)
plt.hist(no_attr['Age'],  bins=20, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['Age'], bins=20, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('Age vs Attrition')
plt.legend()

plt.subplot(2, 3, 2)
plt.hist(no_attr['MonthlyIncome'],  bins=20, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['MonthlyIncome'], bins=20, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('MonthlyIncome vs Attrition')
plt.legend()

plt.subplot(2, 3, 3)
plt.hist(no_attr['TotalWorkingYears'],  bins=20, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['TotalWorkingYears'], bins=20, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('TotalWorkingYears vs Attrition')
plt.legend()

plt.subplot(2, 3, 4)
plt.hist(no_attr['YearsAtCompany'],  bins=20, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['YearsAtCompany'], bins=20, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('YearsAtCompany vs Attrition')
plt.legend()

plt.subplot(2, 3, 5)
plt.hist(no_attr['DistanceFromHome'],  bins=20, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['DistanceFromHome'], bins=20, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('DistanceFromHome vs Attrition')
plt.legend()

plt.subplot(2, 3, 6)
plt.hist(no_attr['YearsSinceLastPromotion'],  bins=15, color='steelblue', alpha=0.6, edgecolor='white', label='No')
plt.hist(yes_attr['YearsSinceLastPromotion'], bins=15, color='tomato',    alpha=0.6, edgecolor='white', label='Yes')
plt.title('YearsSinceLastPromotion vs Attrition')
plt.legend()

plt.suptitle("Bivariate Analysis - Numerical Features vs Attrition", fontsize=14)
plt.tight_layout()
plt.savefig("output_plots/06_bivariate_numerical_vs_attrition.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 06_bivariate_numerical_vs_attrition.png")



[Saved] 06_bivariate_numerical_vs_attrition.png


In [106]:
# correlation heatmap
plt.figure(figsize=(18, 14))
sns.heatmap(df[num_cols].corr(), annot=True, fmt=".2f", cmap="coolwarm", linewidths=0.5)
plt.title("Correlation Heatmap - Numerical Features")
plt.tight_layout()
plt.savefig("output_plots/07_correlation_heatmap.png", dpi=150, bbox_inches='tight')
plt.show()
print("[Saved] 07_correlation_heatmap.png")


[Saved] 07_correlation_heatmap.png


In [107]:
# Feature Engineering
# Convert text columns to numbers so ML model can understand them

# making copy of original df, so that original remains same.
df_model = df.copy()

# encoding target: Yes = 1, No = 0
df_model['Attrition'] = df_model['Attrition'].map({'Yes': 1, 'No': 0})
print("Attrition encoded: Yes=1, No=0")

# encoding binary columns using map
df_model['Gender']   = df_model['Gender'].map({'Male': 1, 'Female': 0})
df_model['OverTime'] = df_model['OverTime'].map({'Yes': 1, 'No': 0})
print("Gender and OverTime encoded")


Attrition encoded: Yes=1, No=0
Gender and OverTime encoded


In [108]:
# ordinal encoding for BusinessTravel (has a natural order: no travel , rarely , frequently)

df_model['BusinessTravel'] = df_model['BusinessTravel'].map(
    {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
)
print("BusinessTravel encoded as ordinal")

BusinessTravel encoded as ordinal


In [109]:
# one-hot encoding for remaining nominal columns (no natural order between categories)
# drop_first=True avoids the dummy variable trap / multicollinearity

df_model = pd.get_dummies(df_model, columns=['Department', 'EducationField', 'JobRole', 'MaritalStatus'], drop_first=True)
print("One-hot encoding done. Shape:", df_model.shape)

# X = input features, y = output
X = df_model.drop(columns=['Attrition'])
y = df_model['Attrition']


One-hot encoding done. Shape: (1470, 44)


In [110]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("Class distribution:\n", y.value_counts())

X shape: (1470, 43)
y shape: (1470,)
Class distribution:
 Attrition
0    1233
1     237
Name: count, dtype: int64


In [111]:
# 80% for training, 20% for testing
# stratify=y keeps the same Yes/No ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\nX_train:", X_train.shape)
print("X_test :", X_test.shape)


X_train: (1176, 43)
X_test : (294, 43)


In [112]:
# StandardScaler: brings all features to the same scale
# We fit only on train data to avoid data leakage into test

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)
print("Scaling done.")

Scaling done.


In [113]:
# Model Building
# Training 5 different classification models one by one
# class_weight='balanced' handles class imbalance automatically

print("\nSTEP 5: MODEL BUILDING")

# Model 1: Logistic Regression
# Good simple baseline model for binary classification problems
lr = LogisticRegression(max_iter=1000, class_weight='balanced')
lr.fit(X_train_scaled, y_train)
print("Logistic Regression trained")

# Model 2: Decision Tree
# Easy to explain to non-technical stakeholders; shows decision rules visually
dt = DecisionTreeClassifier(max_depth=20, class_weight='balanced')
dt.fit(X_train_scaled, y_train)
print("Decision Tree trained")

# Model 3: Random Forest
# Combines many decision trees; usually gives best accuracy + feature importance
rf = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced')
rf.fit(X_train_scaled, y_train)
print("Random Forest trained")

# Model 4: KNN (K-Nearest Neighbors)
# Classifies based on similarity to nearest training examples
knn = KNeighborsClassifier(n_neighbors=7)
knn.fit(X_train_scaled, y_train)
print("KNN trained")

# Model 5: Naive Bayes
# Fast probabilistic model; assumes all features are independent of each other
nb = GaussianNB()
nb.fit(X_train_scaled, y_train)
print("Naive Bayes trained")




STEP 5: MODEL BUILDING
Logistic Regression trained
Decision Tree trained
Random Forest trained
KNN trained
Naive Bayes trained


In [114]:
# 5-Fold Stratified Cross Validation

cv     = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ['accuracy', 'precision', 'recall', 'f1']

models = {
    'Logistic Regression': lr,
    'Decision Tree'      : dt,
    'Random Forest'      : rf,
    'KNN'                : knn,
    'Naive Bayes'        : nb
}

cv_results = []
for name, model in models.items():
    scores = cross_validate(model, X_train_scaled, y_train, cv=cv, scoring=scoring)
    cv_results.append({
        'Model'    : name,
        'Accuracy' : scores['test_accuracy'].mean(),
        'Precision': scores['test_precision'].mean(),
        'Recall'   : scores['test_recall'].mean(),
        'F1-Score' : scores['test_f1'].mean()
    })
    print(name, "-> F1:", round(scores['test_f1'].mean(), 4))

cv_df = pd.DataFrame(cv_results).sort_values('F1-Score', ascending=False).reset_index(drop=True)
print("\nCross Validation Results (averaged over 5 folds):")
print(cv_df.round(4))


Logistic Regression -> F1: 0.4856
Decision Tree -> F1: 0.2705
Random Forest -> F1: 0.297
KNN -> F1: 0.1895
Naive Bayes -> F1: 0.388

Cross Validation Results (averaged over 5 folds):
                 Model  Accuracy  Precision  Recall  F1-Score
0  Logistic Regression    0.7491     0.3650  0.7316    0.4856
1          Naive Bayes    0.5934     0.2640  0.7684    0.3880
2        Random Forest    0.8622     0.8492  0.1842    0.2970
3        Decision Tree    0.7746     0.2927  0.2579    0.2705
4                  KNN    0.8435     0.5571  0.1158    0.1895


In [115]:
# Hyperparameter Tuning
# GridSearchCV tries every combination of parameters and picks the best one (by F1-score)
# cv=3 keeps it fast; scoring='f1' is preferred for imbalanced classes

# --- Logistic Regression ---
lr_params = {'C': [0.01, 0.1, 1, 10]}
lr_grid   = GridSearchCV(LogisticRegression(max_iter=1000, class_weight='balanced'),
lr_params, cv=3, scoring='f1', n_jobs=1)
lr_grid.fit(X_train_scaled, y_train)
lr = lr_grid.best_estimator_
print("Logistic Regression best params:", lr_grid.best_params_)

# --- Decision Tree ---
dt_params = {'max_depth': [5, 10, 15, 20], 'min_samples_split': [2, 5, 10]}
dt_grid   = GridSearchCV(DecisionTreeClassifier(class_weight='balanced'),
            dt_params, cv=3, scoring='f1', n_jobs=1)
dt_grid.fit(X_train_scaled, y_train)
dt = dt_grid.best_estimator_
print("Decision Tree best params:", dt_grid.best_params_)

# --- Random Forest ---
rf_params = {'n_estimators': [50, 100], 'max_depth': [5, 10, 15]}
rf_grid   = GridSearchCV(RandomForestClassifier(class_weight='balanced', random_state=42),
            rf_params, cv=3, scoring='f1', n_jobs=1)
rf_grid.fit(X_train_scaled, y_train)
rf = rf_grid.best_estimator_
print("Random Forest best params:", rf_grid.best_params_)

# --- KNN ---
knn_params = {'n_neighbors': [3, 5, 7, 9, 11]}
knn_grid   = GridSearchCV(KNeighborsClassifier(),
            knn_params, cv=3, scoring='f1', n_jobs=1)
knn_grid.fit(X_train_scaled, y_train)
knn = knn_grid.best_estimator_
print("KNN best params:", knn_grid.best_params_)

# --- Naive Bayes ---
nb_params = {'var_smoothing': [1e-9, 1e-8, 1e-7, 1e-6]}
nb_grid   = GridSearchCV(GaussianNB(), nb_params, cv=3, scoring='f1', n_jobs=1)
nb_grid.fit(X_train_scaled, y_train)
nb = nb_grid.best_estimator_
print("Naive Bayes best params:", nb_grid.best_params_)

print("\nAll models retrained with best hyperparameters.")


Logistic Regression best params: {'C': 0.1}
Decision Tree best params: {'max_depth': 5, 'min_samples_split': 5}
Random Forest best params: {'max_depth': 5, 'n_estimators': 50}
KNN best params: {'n_neighbors': 3}
Naive Bayes best params: {'var_smoothing': 1e-06}

All models retrained with best hyperparameters.


In [116]:
# update models dict so CV results reference the tuned models
models = {
    'Logistic Regression': lr,
    'Decision Tree'      : dt,
    'Random Forest'      : rf,
    'KNN'                : knn,
    'Naive Bayes'        : nb
}


In [117]:
# Model Evaluation
# Compare all 5 models using Accuracy, Precision, Recall, F1-Score
# F1-Score is the most important metric here because data is imbalanced
# =============================================================================

y_pred_lr  = lr.predict(X_test_scaled)
y_pred_dt  = dt.predict(X_test_scaled)
y_pred_rf  = rf.predict(X_test_scaled)
y_pred_knn = knn.predict(X_test_scaled)
y_pred_nb  = nb.predict(X_test_scaled)

#  Logistic Regression
print("\n--- Logistic Regression ---")
print("Accuracy :", accuracy_score(y_test, y_pred_lr))
print("Precision:", precision_score(y_test, y_pred_lr))
print("Recall   :", recall_score(y_test, y_pred_lr))
print("F1-Score :", f1_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr, target_names=['No Attrition', 'Attrition']))

cm_lr = confusion_matrix(y_test, y_pred_lr)
print("Confusion Matrix:\n", cm_lr)

# --- Decision Tree ---
print("\n--- Decision Tree ---")
print("Accuracy :", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall   :", recall_score(y_test, y_pred_dt))
print("F1-Score :", f1_score(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt, target_names=['No Attrition', 'Attrition']))

cm_dt = confusion_matrix(y_test, y_pred_dt)
print("Confusion Matrix:\n", cm_dt)


# --- Random Forest ---
print("\n--- Random Forest ---")
print("Accuracy :", accuracy_score(y_test, y_pred_rf))
print("Precision:", precision_score(y_test, y_pred_rf))
print("Recall   :", recall_score(y_test, y_pred_rf))
print("F1-Score :", f1_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf, target_names=['No Attrition', 'Attrition']))

cm_rf = confusion_matrix(y_test, y_pred_rf)
print("Confusion Matrix:\n", cm_rf)


# --- KNN ---
print("\n--- KNN ---")
print("Accuracy :", accuracy_score(y_test, y_pred_knn))
print("Precision:", precision_score(y_test, y_pred_knn))
print("Recall   :", recall_score(y_test, y_pred_knn))
print("F1-Score :", f1_score(y_test, y_pred_knn))
print(classification_report(y_test, y_pred_knn, target_names=['No Attrition', 'Attrition']))

cm_knn = confusion_matrix(y_test, y_pred_knn)
print("Confusion Matrix:\n", cm_knn)


# --- Naive Bayes ---
print("\n--- Naive Bayes ---")
print("Accuracy :", accuracy_score(y_test, y_pred_nb))
print("Precision:", precision_score(y_test, y_pred_nb))
print("Recall   :", recall_score(y_test, y_pred_nb))
print("F1-Score :", f1_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb, target_names=['No Attrition', 'Attrition']))

cm_nb = confusion_matrix(y_test, y_pred_nb)
print("Confusion Matrix:\n", cm_nb)



--- Logistic Regression ---
Accuracy : 0.7687074829931972
Precision: 0.3764705882352941
Recall   : 0.6808510638297872
F1-Score : 0.48484848484848486
              precision    recall  f1-score   support

No Attrition       0.93      0.79      0.85       247
   Attrition       0.38      0.68      0.48        47

    accuracy                           0.77       294
   macro avg       0.65      0.73      0.67       294
weighted avg       0.84      0.77      0.79       294

Confusion Matrix:
 [[194  53]
 [ 15  32]]

--- Decision Tree ---
Accuracy : 0.7653061224489796
Precision: 0.3472222222222222
Recall   : 0.5319148936170213
F1-Score : 0.42016806722689076
              precision    recall  f1-score   support

No Attrition       0.90      0.81      0.85       247
   Attrition       0.35      0.53      0.42        47

    accuracy                           0.77       294
   macro avg       0.62      0.67      0.64       294
weighted avg       0.81      0.77      0.78       294

Confusion 

In [118]:
# --- Model Comparison ---
print("\n--- Model Comparison Summary ---")
comparison = {
    'Model': ['Logistic Regression', 'Decision Tree', 'Random Forest', 'KNN', 'Naive Bayes'],
    'Accuracy': [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_dt),
        accuracy_score(y_test, y_pred_rf),
        accuracy_score(y_test, y_pred_knn),
        accuracy_score(y_test, y_pred_nb)
    ],
    'Precision': [
        precision_score(y_test, y_pred_lr),
        precision_score(y_test, y_pred_dt),
        precision_score(y_test, y_pred_rf),
        precision_score(y_test, y_pred_knn),
        precision_score(y_test, y_pred_nb)
    ],
    'Recall': [
        recall_score(y_test, y_pred_lr),
        recall_score(y_test, y_pred_dt),
        recall_score(y_test, y_pred_rf),
        recall_score(y_test, y_pred_knn),
        recall_score(y_test, y_pred_nb)
    ],
    'F1-Score': [
        f1_score(y_test, y_pred_lr),
        f1_score(y_test, y_pred_dt),
        f1_score(y_test, y_pred_rf),
        f1_score(y_test, y_pred_knn),
        f1_score(y_test, y_pred_nb)
    ]
}

comparison_df = pd.DataFrame(comparison)
comparison_df = comparison_df.sort_values('F1-Score', ascending=False).reset_index(drop=True)
print(comparison_df.round(4))



--- Model Comparison Summary ---
                 Model  Accuracy  Precision  Recall  F1-Score
0  Logistic Regression    0.7687     0.3765  0.6809    0.4848
1        Random Forest    0.8333     0.4783  0.4681    0.4731
2        Decision Tree    0.7653     0.3472  0.5319    0.4202
3          Naive Bayes    0.6224     0.2538  0.7021    0.3729
4                  KNN    0.8197     0.3125  0.1064    0.1587


In [119]:
best_row = comparison_df.iloc[0]
print("\nBest Model (by F1-Score):", best_row['Model'])
print("F1-Score :", round(best_row['F1-Score'], 4))
print("Accuracy :", round(best_row['Accuracy'], 4))




Best Model (by F1-Score): Logistic Regression
F1-Score : 0.4848
Accuracy : 0.7687


In [120]:
# Feature Importance (using Random Forest)

feature_names = X.columns.tolist()

importance_scores = rf.feature_importances_

feat_imp_df = pd.DataFrame({
    'Feature'   : feature_names,
    'Importance': importance_scores
})

feat_imp_df = feat_imp_df.sort_values('Importance', ascending=False)

print("\nTop Features that influence Attrition the most:")
print(feat_imp_df.round(4))



Top Features that influence Attrition the most:
                              Feature  Importance
19                  TotalWorkingYears      0.0877
14                           OverTime      0.0844
0                                 Age      0.0758
11                      MonthlyIncome      0.0706
25               YearsWithCurrManager      0.0650
22                     YearsAtCompany      0.0583
18                   StockOptionLevel      0.0417
23                 YearsInCurrentRole      0.0407
3                    DistanceFromHome      0.0375
2                           DailyRate      0.0346
9                            JobLevel      0.0346
13                 NumCompaniesWorked      0.0327
7                          HourlyRate      0.0303
10                    JobSatisfaction      0.0272
21                    WorkLifeBalance      0.0218
5             EnvironmentSatisfaction      0.0196
15                  PercentSalaryHike      0.0193
42               MaritalStatus_Single      0.0187
1

In [122]:
#  Predict for New / Sample Employees

# taking 2 sample employees from the test set for demonstration
sample_X      = X_test_scaled[:2]
sample_actual = y_test.iloc[:2].values

# use model which won the comparison (highest F1-Score)
model_map = {
    'Logistic Regression': lr,
    'Decision Tree'      : dt,
    'Random Forest'      : rf,
    'KNN'                : knn,
    'Naive Bayes'        : nb
}
best_model      = model_map[best_row['Model']]
sample_pred     = best_model.predict(sample_X)
sample_probability    = best_model.predict_proba(sample_X)[:, 1]
print("\nPredictions using best model:", best_row['Model'])



Predictions using best model: Logistic Regression


In [124]:
## Testing with 2 employee

print("Employee 1:")
print("  Actual    :", "Attrition" if sample_actual[0] == 1 else "No Attrition")
print("  Predicted :", "Attrition" if sample_pred[0] == 1   else "No Attrition")
print("  Probability:", round(sample_probability[0] * 100, 2), "%")

print("Employee 2:")
print("  Actual    :", "Attrition" if sample_actual[1] == 1 else "No Attrition")
print("  Predicted :", "Attrition" if sample_pred[1] == 1   else "No Attrition")
print("  Probability:", round(sample_probability[1] * 100, 2), "%")

print("\nAll plots saved to:", os.path.abspath("output_plots"))


Employee 1:
  Actual    : No Attrition
  Predicted : No Attrition
  Probability: 34.79 %
Employee 2:
  Actual    : No Attrition
  Predicted : No Attrition
  Probability: 2.51 %

All plots saved to: /content/output_plots
